# Chess Piece Classifier — TFLite Model (v4)

**Key improvements over v3:**
- ✅ Training images are converted to **grayscale** (same as inference from PDF pages)
- ✅ Each piece rendered on both **light and dark** squares
- ✅ Multiple **board styles** (classic B&W, brown, gray tones)
- ✅ **Augmentation**: brightness, contrast, blur, Gaussian noise
- ✅ ~400 samples per class
- ✅ Two-phase training: frozen base → fine-tune top layers
- ✅ **Confusion matrix** to diagnose per-class errors

**Output:** `chess_pieces.tflite` (~3-5 MB)

## Step 1: Install Dependencies

In [ ]:
!apt-get update -qq && apt-get install -qq libcairo2-dev pkg-config python3-dev
!pip install -q chess tensorflow pillow numpy matplotlib scikit-learn cairosvg scipy

## Step 2: Imports

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter, ImageEnhance
from io import BytesIO
import chess
import chess.svg
import cairosvg
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {len(tf.config.list_physical_devices('GPU'))} device(s)")
print(f"✅ python-chess: installed")

## Step 3: Define Classes and Board Styles

In [ ]:
CLASS_NAMES = [
    'empty',
    'wP', 'wN', 'wB', 'wR', 'wQ', 'wK',
    'bP', 'bN', 'bB', 'bR', 'bQ', 'bK',
]

PIECE_MAP = {
    'wP': (chess.PAWN,   True),
    'wN': (chess.KNIGHT, True),
    'wB': (chess.BISHOP, True),
    'wR': (chess.ROOK,   True),
    'wQ': (chess.QUEEN,  True),
    'wK': (chess.KING,   True),
    'bP': (chess.PAWN,   False),
    'bN': (chess.KNIGHT, False),
    'bB': (chess.BISHOP, False),
    'bR': (chess.ROOK,   False),
    'bQ': (chess.QUEEN,  False),
    'bK': (chess.KING,   False),
}

# Board styles: (light_rgb, dark_rgb)
# These mimic printed chess book styles (all convert to grayscale anyway)
BOARD_STYLES = [
    ((255, 255, 255), (180, 180, 180)),  # Classic B&W
    ((240, 217, 181), (181, 136, 99)),   # Lichess brown
    ((230, 230, 230), (120, 120, 120)),  # Soft gray
    ((255, 255, 255), (90,  90,  90)),   # High-contrast
]

print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
print(f"Board styles: {len(BOARD_STYLES)}")

## Step 4: Rendering Function (Grayscale Output)

**Critical fix from v3:** After rendering, we convert to grayscale and store as `[v, v, v]` — exactly what `board_detector.dart` sends to the model at inference time.

In [ ]:
_RENDER_CACHE = {}  # Cache SVG renders to avoid re-rendering identical pieces

def render_square_gray(class_name, is_light_square, board_style, size=64,
                        add_hatching=False):
    """
    Render a chess square and return a grayscale numpy array of shape (64, 64, 3)
    with identical values in all 3 channels — matching the format used by
    board_detector.dart during inference.
    """
    from PIL import ImageDraw
    light_col, dark_col = board_style
    bg_color = light_col if is_light_square else dark_col
    img = Image.new('RGB', (size, size), bg_color)

    # Optional hatching on dark squares (some printed books use this)
    if not is_light_square and add_hatching:
        draw = ImageDraw.Draw(img)
        line_color = tuple(max(0, c - 40) for c in bg_color)
        for i in range(-size, size * 2, 5):
            draw.line([(i, 0), (i + size, size)], fill=line_color, width=1)

    if class_name != 'empty':
        piece_type, is_white = PIECE_MAP[class_name]
        cache_key = (class_name,)
        if cache_key not in _RENDER_CACHE:
            board = chess.Board()
            board.clear()
            board.set_piece_at(chess.A1, chess.Piece(piece_type, is_white))
            render_size = 1024
            svg_str = chess.svg.board(board, size=render_size)
            png_bytes = cairosvg.svg2png(
                bytestring=svg_str.encode('utf-8'),
                output_width=render_size,
                output_height=render_size,
            )
            board_img = Image.open(BytesIO(png_bytes)).convert('RGB')
            board_arr = np.array(board_img)
            margin = int(render_size * 0.038)
            playable = board_arr[margin:render_size - margin,
                                 margin:render_size - margin]
            h, w = playable.shape[:2]
            sq_w, sq_h = w // 8, h // 8
            # a1 square = bottom-left (rank 0, file 0 in python-chess)
            piece_arr = playable[7 * sq_h:8 * sq_h, 0:sq_w]
            _RENDER_CACHE[cache_key] = piece_arr

        piece_arr = _RENDER_CACHE[cache_key]
        piece_img = Image.fromarray(piece_arr).resize((size, size), Image.Resampling.LANCZOS)

        # Composite the piece over the background
        # The SVG-rendered piece has a white background we need to blend away
        piece_rgba = piece_img.convert('RGBA')
        bg = img.convert('RGBA')
        # Simple blend: mix piece pixels (white bg) with our background color
        piece_np = np.array(piece_rgba).astype(float)
        bg_np = np.array(bg).astype(float)
        # Treat white (>240) as transparent in piece
        white_mask = (piece_np[:,:,0] > 240) & (piece_np[:,:,1] > 240) & (piece_np[:,:,2] > 240)
        blended = piece_np.copy()
        blended[white_mask] = bg_np[white_mask]
        img = Image.fromarray(blended[:,:,:3].astype(np.uint8))

    # ── CRITICAL FIX ── Convert to grayscale, then back to RGB with equal channels
    # This matches exactly what board_detector.dart produces:
    #   sq[idx++] = v; sq[idx++] = v; sq[idx++] = v;
    gray = img.convert('L')  # true grayscale
    gray_np = np.array(gray, dtype=np.float32) / 255.0  # shape (64, 64)
    rgb_gray = np.stack([gray_np, gray_np, gray_np], axis=-1)  # shape (64, 64, 3)
    return rgb_gray

print("✅ Render function ready (grayscale output matching inference)")

## Step 5: Augmentation Function

In [ ]:
def augment(img_arr, rng):
    """
    Apply random augmentation to a (64, 64, 3) float32 grayscale image.
    Returns augmented array of same shape.
    """
    # Convert back to PIL for augmentation
    uint8 = (img_arr[:, :, 0] * 255).clip(0, 255).astype(np.uint8)
    pil = Image.fromarray(uint8, mode='L')

    # Brightness ±20%
    factor = rng.uniform(0.80, 1.20)
    pil = ImageEnhance.Brightness(pil).enhance(factor)

    # Contrast ±20%
    factor = rng.uniform(0.80, 1.20)
    pil = ImageEnhance.Contrast(pil).enhance(factor)

    # Random slight blur (50% chance)
    if rng.random() < 0.5:
        radius = rng.uniform(0.3, 1.2)
        pil = pil.filter(ImageFilter.GaussianBlur(radius=radius))

    # Gaussian noise
    arr = np.array(pil, dtype=np.float32) / 255.0
    noise = rng.normal(0, 0.02, arr.shape).astype(np.float32)
    arr = np.clip(arr + noise, 0.0, 1.0)

    return np.stack([arr, arr, arr], axis=-1)

print("✅ Augmentation function ready")

## Step 6: Test Rendering — Verify Grayscale Output

In [ ]:
fig, axes = plt.subplots(2, len(CLASS_NAMES), figsize=(len(CLASS_NAMES) * 1.5, 4))

style = BOARD_STYLES[0]
for col, cls in enumerate(CLASS_NAMES):
    for row, is_light in enumerate([True, False]):
        img = render_square_gray(cls, is_light, style)
        axes[row, col].imshow(img[:, :, 0], cmap='gray', vmin=0, vmax=1)
        if row == 0:
            axes[row, col].set_title(cls, fontsize=8)
        axes[row, col].axis('off')

axes[0, 0].set_ylabel('Light', fontsize=8)
axes[1, 0].set_ylabel('Dark', fontsize=8)
plt.suptitle('All pieces — Grayscale (top=light sq, bottom=dark sq)')
plt.tight_layout()
plt.show()

# Verify the 3 channels are truly identical
test = render_square_gray('wK', True, BOARD_STYLES[0])
assert np.allclose(test[:,:,0], test[:,:,1]) and np.allclose(test[:,:,0], test[:,:,2]), \
    "ERROR: channels are NOT equal — grayscale fix failed!"
print(f"✅ Channels are identical (shape={test.shape})")
print("✅ All pieces rendered correctly")

## Step 7: Generate Training Dataset

- 4 board styles × 2 square colors × 50 augmentations = **400 samples per class**
- Total: 13 classes × 400 = **5 200 samples**

In [ ]:
AUGMENTATIONS_PER_BASE = 50

def generate_dataset(augmentations_per_base=AUGMENTATIONS_PER_BASE, img_size=64, seed=42):
    rng = np.random.default_rng(seed)
    X, y = [], []

    base_configs = [
        (style, is_light, add_hatch)
        for style in BOARD_STYLES
        for is_light in [True, False]
        for add_hatch in [False, True]
    ]

    total_classes = len(CLASS_NAMES)
    for class_idx, class_name in enumerate(CLASS_NAMES):
        count = 0
        for style, is_light, add_hatch in base_configs:
            base_img = render_square_gray(class_name, is_light, style,
                                          size=img_size, add_hatching=add_hatch)
            for _ in range(augmentations_per_base):
                aug = augment(base_img, rng)
                X.append(aug)
                y.append(class_idx)
                count += 1

        print(f"  [{class_idx+1:2d}/{total_classes}] {class_name:6s}: {count} samples")

    print(f"\n✅ Total: {len(X)} samples across {total_classes} classes")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

print("Generating dataset (may take 2-3 minutes)...")
X, y = generate_dataset()
print(f"Dataset shape: X={X.shape}  y={y.shape}")

## Step 8: Split Data

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

BATCH = 32
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(2000).batch(BATCH)
val_ds   = tf.data.Dataset.from_tensor_slices((X_val,   y_val  )).batch(BATCH)
test_ds  = tf.data.Dataset.from_tensor_slices((X_test,  y_test )).batch(BATCH)

print("✅ Data split and batched")

## Step 9: Build Model (MobileNetV2)

In [ ]:
def build_model(num_classes=13, trainable_base=False):
    base = keras.applications.MobileNetV2(
        input_shape=(64, 64, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = trainable_base

    model = keras.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model, base

model, base_model = build_model(num_classes=len(CLASS_NAMES), trainable_base=False)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

print("✅ Model compiled (Phase 1: frozen base)")
print(f"   Trainable params: {sum(np.prod(v.shape) for v in model.trainable_variables):,}")

## Step 10: Phase 1 Training — Frozen Base

In [ ]:
callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
    ),
]

print("=== Phase 1: Frozen base ===")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_phase1,
    verbose=1,
)
val_acc1 = max(history1.history['val_accuracy'])
print(f"\nPhase 1 best val accuracy: {val_acc1:.2%}")

## Step 11: Phase 2 Training — Fine-tune Top Layers

In [ ]:
# Unfreeze the top 50 layers of MobileNetV2 for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # Lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

trainable_count = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"Phase 2 trainable params: {trainable_count:,}")

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, mode='max'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
    ),
]

print("=== Phase 2: Fine-tuning top layers ===")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_phase2,
    verbose=1,
)
val_acc2 = max(history2.history['val_accuracy'])
print(f"\nPhase 2 best val accuracy: {val_acc2:.2%}")

## Step 12: Evaluate & Training Curves

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\n📊 Test Loss:     {test_loss:.4f}")
print(f"🎯 Test Accuracy: {test_acc:.2%}")

# Combine histories for plotting
all_loss     = history1.history['loss']     + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
all_acc      = history1.history['accuracy']     + history2.history['accuracy']
all_val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
phase_split  = len(history1.history['loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(all_loss, label='train')
ax1.plot(all_val_loss, label='val')
ax1.axvline(phase_split, color='gray', linestyle='--', label='fine-tune start')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(all_acc, label='train')
ax2.plot(all_val_acc, label='val')
ax2.axvline(phase_split, color='gray', linestyle='--', label='fine-tune start')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 13: Confusion Matrix

Useful for diagnosing which pieces are confused with each other (e.g., Bishop vs King).

In [ ]:
y_pred_probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm_norm,
    annot=True, fmt='.0%',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cmap='Blues', ax=ax, vmin=0, vmax=1,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (row-normalized) — diagonal = correct')
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, cls in enumerate(CLASS_NAMES):
    acc = cm_norm[i, i]
    bar = '█' * int(acc * 20)
    print(f"  {cls:6s}: {acc:5.1%}  {bar}")

## Step 14: Convert to TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
model_path = 'chess_pieces.tflite'
with open(model_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"\n✅ Model saved: {model_path}")
print(f"   Size: {size_mb:.2f} MB")

## Step 15: Verify TFLite Model

In [ ]:
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("🔧 TFLite Model Info:")
print(f"   Input shape:  {input_details[0]['shape']}")
print(f"   Output shape: {output_details[0]['shape']}")

# Quick sanity check: run the TFLite model on all test classes
correct = 0
for class_name in CLASS_NAMES:
    sample = render_square_gray(class_name, True, BOARD_STYLES[0])[np.newaxis]  # (1, 64, 64, 3)
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    pred_idx = int(np.argmax(output[0]))
    pred_name = CLASS_NAMES[pred_idx]
    ok = '✅' if pred_name == class_name else '❌'
    print(f"  {ok} {class_name:6s} → predicted: {pred_name:6s}  ({output[0][pred_idx]:.1%})")
    if pred_name == class_name:
        correct += 1

print(f"\nSanity check: {correct}/{len(CLASS_NAMES)} correct ({correct/len(CLASS_NAMES):.0%})")

## Step 16: Download Model

🎉 **Your model is ready!**

### Setup in Flutter:

1. **Download** `chess_pieces.tflite` from the Files panel (📁 on the left)
2. **Place** it at `assets/models/chess_pieces.tflite` in the Flutter project
3. **Run:** `flutter run`

### What changed from v3:
- ✅ Grayscale training matches grayscale inference from PDF pixels
- ✅ Both light and dark squares represented in training data
- ✅ 4 board styles (B&W, brown, gray) for robustness
- ✅ Augmentation (brightness/contrast/blur/noise) reduces overfitting
- ✅ Two-phase training: frozen base first, then fine-tuning
- ✅ Confusion matrix to verify no class is systematically confused

In [ ]:
print(f"""
╔═══════════════════════════════════════════════╗
║   ✅ Model v4 Ready for Download               ║
╚═══════════════════════════════════════════════╝

📦 File:     chess_pieces.tflite
📊 Size:     {size_mb:.2f} MB
🎯 Accuracy: {test_acc:.2%}
🎨 Classes:  {len(CLASS_NAMES)}
🖤 Mode:     Grayscale (matches PDF inference)

⬇️  Download from Files panel (📁)
📁 Save to: assets/models/chess_pieces.tflite
▶️  Run: flutter run
""")